# CloudTube Telegram Bot

Run this notebook in **Google Colab** to start your Telegram YouTube uploader bot.

**How it works:**
1. Run all cells (or tap **Runtime → Run all** on mobile)
2. Authenticate with YouTube (one-time — token is saved to Google Drive)
3. The bot starts polling Telegram — message it from your phone!
4. Bot stays alive as long as this Colab session is running (~12 hours)
5. When it dies, just re-run this notebook — your auth token persists on Drive

### Prerequisites
- A Telegram bot token from [@BotFather](https://t.me/BotFather)
- Your Telegram user ID from [@userinfobot](https://t.me/userinfobot)
- A Google account with a YouTube channel

## 1. Install Dependencies

In [ ]:
!pip install -q python-telegram-bot google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client requests nest_asyncio

## 2. Configuration

Fill in your **Telegram bot token** and **user ID** below.

In [ ]:
#@title Bot Configuration { display-mode: "form" }
#@markdown Get your bot token from [@BotFather](https://t.me/BotFather) and user ID from [@userinfobot](https://t.me/userinfobot).
TELEGRAM_BOT_TOKEN = ""  #@param {type:"string"}
ALLOWED_USER_ID = 0  #@param {type:"integer"}

if not TELEGRAM_BOT_TOKEN or not ALLOWED_USER_ID:
    raise ValueError("Please fill in TELEGRAM_BOT_TOKEN and ALLOWED_USER_ID above.")
print(f"Bot token: ...{TELEGRAM_BOT_TOKEN[-6:]}")
print(f"Allowed user: {ALLOWED_USER_ID}")

## 3. Mount Google Drive (for persistent auth token)

Your YouTube auth token is saved to Google Drive so you don't need to re-authenticate every time you restart Colab.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

PERSIST_DIR = '/content/drive/MyDrive/CloudTubeBot'
os.makedirs(PERSIST_DIR, exist_ok=True)

TOKEN_PATH = os.path.join(PERSIST_DIR, 'token.pickle')
print(f"Token will be saved to: {TOKEN_PATH}")

## 4. Load OAuth Credentials

In [ ]:
CREDS_URL = "https://raw.githubusercontent.com/mathinraj/CloudTube/main/client_secret.json"
CLIENT_SECRET_PATH = 'client_secret.json'

if not os.path.exists(CLIENT_SECRET_PATH):
    print("Fetching bundled credentials from repo...")
    os.system(f'wget -q -O {CLIENT_SECRET_PATH} "{CREDS_URL}"')

if os.path.exists(CLIENT_SECRET_PATH) and os.path.getsize(CLIENT_SECRET_PATH) > 0:
    print("Credentials loaded.")
else:
    raise FileNotFoundError("client_secret.json not found.")

## 5. Authenticate with YouTube

If you've already authenticated before, the saved token will be reused automatically. Otherwise, follow the link to authorize.

In [ ]:
import pickle
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

SCOPES = [
    'https://www.googleapis.com/auth/youtube.upload',
    'https://www.googleapis.com/auth/youtube',
    'https://www.googleapis.com/auth/drive.readonly',
]

def authenticate():
    creds = None
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                CLIENT_SECRET_PATH, SCOPES,
                redirect_uri='http://localhost:1',
            )
            auth_url, _ = flow.authorization_url(prompt='consent', access_type='offline')
            print("1. Open this URL in your browser:\n")
            print(auth_url)
            print("\n2. Sign in and grant access.")
            print("3. You'll be redirected to a page that WON'T load — that's expected.")
            print("4. Copy the FULL URL from your browser's address bar and paste it below.\n")
            redirect_response = input("Paste the full redirect URL here: ").strip()
            flow.fetch_token(authorization_response=redirect_response)
            creds = flow.credentials
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)

    return creds

creds = authenticate()
youtube_service = build('youtube', 'v3', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)
print("\n✅ Authenticated successfully! Token saved to Google Drive.")

## 6. Start the Telegram Bot

This cell runs the bot. It will keep running until the Colab session expires.

**Bot commands:**
- `/start` — Welcome + help
- `/upload_url` — Upload from a direct video URL
- `/upload_drive` — Upload from Google Drive link
- `/batch_drive` — Upload all videos from a Drive folder
- `/quota` — YouTube API quota info
- `/cancel` — Cancel current operation

In [ ]:
import asyncio
import logging
import re
import tempfile
import time
from urllib.parse import urlparse

import requests
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import (
    Application, CommandHandler, CallbackQueryHandler,
    ConversationHandler, MessageHandler, filters, ContextTypes,
)
from telegram.constants import ParseMode

logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', level=logging.INFO
)
logger = logging.getLogger(__name__)

# --- Settings ---
CHUNK_SIZE = 10 * 1024 * 1024
MAX_RETRIES = 5
PROGRESS_INTERVAL = 5  # seconds between progress edits
VALID_PRIVACY = ['private', 'unlisted', 'public']
DEFAULT_CATEGORY_ID = '22'
TEMP_DIR = '/tmp'

YOUTUBE_CATEGORIES = {
    '1': 'Film & Animation', '2': 'Autos & Vehicles', '10': 'Music',
    '15': 'Pets & Animals', '17': 'Sports', '18': 'Short Movies',
    '19': 'Travel & Events', '20': 'Gaming', '21': 'Videoblogging',
    '22': 'People & Blogs', '23': 'Comedy', '24': 'Entertainment',
    '25': 'News & Politics', '26': 'Howto & Style', '27': 'Education',
    '28': 'Science & Technology', '29': 'Nonprofits & Activism',
}

# --- Conversation states ---
(
    URL_WAITING_LINK, URL_WAITING_TITLE, URL_WAITING_DESC, URL_WAITING_PRIVACY,
    DRIVE_WAITING_LINK, DRIVE_WAITING_TITLE, DRIVE_WAITING_DESC, DRIVE_WAITING_PRIVACY,
    BATCH_WAITING_LINK, BATCH_WAITING_PRIVACY,
) = range(10)


# --- Helpers ---
def extract_drive_file_id(url_or_id: str) -> str:
    for pattern in [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'/open\?id=([a-zA-Z0-9_-]+)',
        r'/folders/([a-zA-Z0-9_-]+)',
    ]:
        match = re.search(pattern, url_or_id)
        if match:
            return match.group(1)
    if re.match(r'^[a-zA-Z0-9_-]{20,}$', url_or_id):
        return url_or_id
    raise ValueError(f"Could not extract Drive file ID from: {url_or_id}")


def privacy_keyboard():
    return InlineKeyboardMarkup([
        [InlineKeyboardButton(p.capitalize(), callback_data=f'privacy:{p}')]
        for p in VALID_PRIVACY
    ])


async def update_progress(msg, text, timer):
    now = time.time()
    if now - timer.get('t', 0) < PROGRESS_INTERVAL:
        return
    timer['t'] = now
    try:
        await msg.edit_text(text)
    except Exception:
        pass


async def yt_upload(media_body, title, description, tags, category_id, privacy, status_msg):
    body = {
        'snippet': {'title': title, 'description': description, 'tags': tags, 'categoryId': category_id},
        'status': {'privacyStatus': privacy, 'selfDeclaredMadeForKids': False},
    }
    request = youtube_service.videos().insert(part='snippet,status', body=body, media_body=media_body)
    response = None
    retries = 0
    timer = {'t': 0}
    while response is None:
        try:
            status, response = request.next_chunk()
            if status:
                pct = int(status.progress() * 100)
                await update_progress(status_msg, f'⬆️ Uploading to YouTube… {pct}%', timer)
        except HttpError as e:
            if e.resp.status in (500, 502, 503, 504) and retries < MAX_RETRIES:
                retries += 1
                await update_progress(status_msg, f'⚠️ Retrying in {2**retries}s…', {'t': 0})
                await asyncio.sleep(2 ** retries)
            else:
                raise
    return response['id']


def authorized(func):
    async def wrapper(update: Update, context: ContextTypes.DEFAULT_TYPE):
        if update.effective_user.id != ALLOWED_USER_ID:
            await update.message.reply_text('Not authorized.')
            return ConversationHandler.END
        return await func(update, context)
    return wrapper


# --- /start ---
@authorized
async def cmd_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        '*CloudTube Telegram Bot*\n\n'
        'Upload videos to YouTube from a direct URL or Google Drive.\n\n'
        '*Commands*\n'
        '/upload\\_url — Upload from a direct video URL\n'
        '/upload\\_drive — Upload from Google Drive\n'
        '/batch\\_drive — Upload all videos from a Drive folder\n'
        '/quota — YouTube API quota info\n'
        '/cancel — Cancel current operation',
        parse_mode=ParseMode.MARKDOWN,
    )


@authorized
async def cmd_quota(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        '*YouTube API Quota*\n\n'
        'Daily quota: 10,000 units\n'
        'Cost per upload: 1,600 units\n'
        'Uploads/day: ~6 videos',
        parse_mode=ParseMode.MARKDOWN,
    )


async def cmd_cancel(update: Update, context: ContextTypes.DEFAULT_TYPE):
    context.user_data.clear()
    await update.message.reply_text('Operation cancelled.')
    return ConversationHandler.END


# --- /upload_url ---
@authorized
async def cmd_upload_url(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text('Send me the direct video URL:')
    return URL_WAITING_LINK

async def url_get_link(update: Update, context: ContextTypes.DEFAULT_TYPE):
    context.user_data['url'] = update.message.text.strip()
    await update.message.reply_text('Video title? (or `.` to use filename)', parse_mode=ParseMode.MARKDOWN)
    return URL_WAITING_TITLE

async def url_get_title(update: Update, context: ContextTypes.DEFAULT_TYPE):
    context.user_data['title'] = update.message.text.strip()
    await update.message.reply_text('Description? (or `.` to skip)')
    return URL_WAITING_DESC

async def url_get_desc(update: Update, context: ContextTypes.DEFAULT_TYPE):
    desc = update.message.text.strip()
    context.user_data['description'] = '' if desc == '.' else desc
    await update.message.reply_text('Choose privacy:', reply_markup=privacy_keyboard())
    return URL_WAITING_PRIVACY

async def url_get_privacy(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    privacy = query.data.split(':')[1]
    ud = context.user_data
    url = ud['url']
    title = ud.get('title', '.')
    description = ud.get('description', '')

    status_msg = await query.message.reply_text('⬇️ Downloading from URL…')
    timer = {'t': 0}
    tmp_path = None

    try:
        resp = requests.get(url, stream=True, timeout=60)
        resp.raise_for_status()
        content_type = resp.headers.get('Content-Type', 'video/mp4')
        content_length = resp.headers.get('Content-Length')

        cd = resp.headers.get('Content-Disposition', '')
        if 'filename=' in cd:
            fname = cd.split('filename=')[-1].strip('\'\"  ')
        else:
            fname = os.path.basename(urlparse(url).path) or 'video.mp4'

        if title == '.':
            title = os.path.splitext(fname)[0]

        tmp_path = os.path.join(TEMP_DIR, fname)
        downloaded = 0
        with open(tmp_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if content_length:
                        pct = int(downloaded / int(content_length) * 100)
                        await update_progress(status_msg, f'⬇️ Downloading… {pct}%', timer)
                    else:
                        mb = downloaded / (1024 * 1024)
                        await update_progress(status_msg, f'⬇️ Downloaded {mb:.1f} MB…', timer)

        await status_msg.edit_text('⬆️ Uploading to YouTube… 0%')
        mime = content_type.split(';')[0].strip()
        if not mime.startswith('video/'):
            mime = 'video/mp4'
        media = MediaFileUpload(tmp_path, mimetype=mime, resumable=True, chunksize=CHUNK_SIZE)
        video_id = await yt_upload(media, title, description, [], DEFAULT_CATEGORY_ID, privacy, status_msg)

        await status_msg.edit_text(
            f'✅ *Upload complete!*\n\n'
            f'Title: {title}\n'
            f'Privacy: {privacy}\n'
            f'[Watch on YouTube](https://www.youtube.com/watch?v={video_id})\n'
            f'[YouTube Studio](https://studio.youtube.com/video/{video_id}/edit)',
            parse_mode=ParseMode.MARKDOWN, disable_web_page_preview=True,
        )
    except Exception as e:
        logger.error('URL upload failed: %s', e)
        await status_msg.edit_text(f'❌ Upload failed: {e}')
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)
        context.user_data.clear()
    return ConversationHandler.END


# --- /upload_drive ---
@authorized
async def cmd_upload_drive(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text('Send me the Google Drive share link or file ID:')
    return DRIVE_WAITING_LINK

async def drive_get_link(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        file_id = extract_drive_file_id(update.message.text.strip())
    except ValueError as e:
        await update.message.reply_text(f'❌ {e}\nTry again or /cancel.')
        return DRIVE_WAITING_LINK
    context.user_data['drive_file_id'] = file_id
    meta = drive_service.files().get(fileId=file_id, fields='name,size,mimeType').execute()
    context.user_data['drive_meta'] = meta
    size_mb = int(meta.get('size', 0)) / (1024 * 1024)
    await update.message.reply_text(
        f'Found: *{meta["name"]}* ({size_mb:.1f} MB)\n\n'
        'Video title? (or `.` to use filename)',
        parse_mode=ParseMode.MARKDOWN,
    )
    return DRIVE_WAITING_TITLE

async def drive_get_title(update: Update, context: ContextTypes.DEFAULT_TYPE):
    context.user_data['title'] = update.message.text.strip()
    await update.message.reply_text('Description? (or `.` to skip)')
    return DRIVE_WAITING_DESC

async def drive_get_desc(update: Update, context: ContextTypes.DEFAULT_TYPE):
    desc = update.message.text.strip()
    context.user_data['description'] = '' if desc == '.' else desc
    await update.message.reply_text('Choose privacy:', reply_markup=privacy_keyboard())
    return DRIVE_WAITING_PRIVACY

async def drive_get_privacy(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    privacy = query.data.split(':')[1]
    ud = context.user_data
    file_id = ud['drive_file_id']
    meta = ud['drive_meta']
    title = ud.get('title', '.')
    description = ud.get('description', '')
    if title == '.':
        title = os.path.splitext(meta['name'])[0]

    status_msg = await query.message.reply_text('⬇️ Downloading from Google Drive…')
    timer = {'t': 0}
    tmp_path = os.path.join(TEMP_DIR, meta['name'])

    try:
        request_dl = drive_service.files().get_media(fileId=file_id)
        with open(tmp_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request_dl, chunksize=CHUNK_SIZE)
            done = False
            while not done:
                dl_status, done = downloader.next_chunk()
                if dl_status:
                    pct = int(dl_status.progress() * 100)
                    await update_progress(status_msg, f'⬇️ Downloading from Drive… {pct}%', timer)

        await status_msg.edit_text('⬆️ Uploading to YouTube… 0%')
        media = MediaFileUpload(tmp_path, mimetype=meta.get('mimeType', 'video/mp4'), resumable=True, chunksize=CHUNK_SIZE)
        video_id = await yt_upload(media, title, description, [], DEFAULT_CATEGORY_ID, privacy, status_msg)

        await status_msg.edit_text(
            f'✅ *Upload complete!*\n\n'
            f'Title: {title}\n'
            f'Privacy: {privacy}\n'
            f'[Watch on YouTube](https://www.youtube.com/watch?v={video_id})\n'
            f'[YouTube Studio](https://studio.youtube.com/video/{video_id}/edit)',
            parse_mode=ParseMode.MARKDOWN, disable_web_page_preview=True,
        )
    except Exception as e:
        logger.error('Drive upload failed: %s', e)
        await status_msg.edit_text(f'❌ Upload failed: {e}')
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        context.user_data.clear()
    return ConversationHandler.END


# --- /batch_drive ---
@authorized
async def cmd_batch_drive(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text('Send me the Google Drive folder link or ID:')
    return BATCH_WAITING_LINK

async def batch_get_link(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        folder_id = extract_drive_file_id(update.message.text.strip())
    except ValueError as e:
        await update.message.reply_text(f'❌ {e}\nTry again or /cancel.')
        return BATCH_WAITING_LINK

    q = f"'{folder_id}' in parents and mimeType contains 'video/' and trashed = false"
    results = drive_service.files().list(q=q, fields='files(id,name,size,mimeType)', orderBy='name', pageSize=100).execute()
    video_files = results.get('files', [])

    if not video_files:
        await update.message.reply_text('No video files found in this folder.')
        return ConversationHandler.END

    listing = '\n'.join(
        f'  {i}. {vf["name"]} ({int(vf.get("size", 0)) / (1024*1024):.1f} MB)'
        for i, vf in enumerate(video_files, 1)
    )
    context.user_data['batch_files'] = video_files
    await update.message.reply_text(
        f'Found *{len(video_files)}* video(s):\n```\n{listing}\n```\n\nChoose privacy for all:',
        parse_mode=ParseMode.MARKDOWN, reply_markup=privacy_keyboard(),
    )
    return BATCH_WAITING_PRIVACY

async def batch_get_privacy(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    await query.answer()
    privacy = query.data.split(':')[1]
    video_files = context.user_data['batch_files']
    total = len(video_files)

    summary_msg = await query.message.reply_text(f'Starting batch upload of {total} video(s)…')
    succeeded, failed = 0, 0

    for i, vf in enumerate(video_files, 1):
        status_msg = await query.message.reply_text(f'⏳ [{i}/{total}] {vf["name"]}')
        tmp_path = os.path.join(TEMP_DIR, vf['name'])
        timer = {'t': 0}
        try:
            await status_msg.edit_text(f'⬇️ [{i}/{total}] Downloading {vf["name"]}…')
            request_dl = drive_service.files().get_media(fileId=vf['id'])
            with open(tmp_path, 'wb') as f:
                downloader = MediaIoBaseDownload(f, request_dl, chunksize=CHUNK_SIZE)
                done = False
                while not done:
                    dl_status, done = downloader.next_chunk()
                    if dl_status:
                        pct = int(dl_status.progress() * 100)
                        await update_progress(status_msg, f'⬇️ [{i}/{total}] Downloading {vf["name"]}… {pct}%', timer)

            title = os.path.splitext(vf['name'])[0]
            media = MediaFileUpload(tmp_path, mimetype=vf.get('mimeType', 'video/mp4'), resumable=True, chunksize=CHUNK_SIZE)
            await status_msg.edit_text(f'⬆️ [{i}/{total}] Uploading {vf["name"]}… 0%')
            video_id = await yt_upload(media, title, '', [], DEFAULT_CATEGORY_ID, privacy, status_msg)
            await status_msg.edit_text(f'✅ [{i}/{total}] {title}\nhttps://www.youtube.com/watch?v={video_id}')
            succeeded += 1
        except Exception as e:
            logger.error('Batch item %s failed: %s', vf['name'], e)
            await status_msg.edit_text(f'❌ [{i}/{total}] {vf["name"]}: {e}')
            failed += 1
        finally:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

    await summary_msg.edit_text(
        f'*Batch complete*\n\n✅ Succeeded: {succeeded}\n❌ Failed: {failed}',
        parse_mode=ParseMode.MARKDOWN,
    )
    context.user_data.clear()
    return ConversationHandler.END


# --- Build & Run ---
cancel_handler = CommandHandler('cancel', cmd_cancel)

url_conv = ConversationHandler(
    entry_points=[CommandHandler('upload_url', cmd_upload_url)],
    states={
        URL_WAITING_LINK: [MessageHandler(filters.TEXT & ~filters.COMMAND, url_get_link)],
        URL_WAITING_TITLE: [MessageHandler(filters.TEXT & ~filters.COMMAND, url_get_title)],
        URL_WAITING_DESC: [MessageHandler(filters.TEXT & ~filters.COMMAND, url_get_desc)],
        URL_WAITING_PRIVACY: [CallbackQueryHandler(url_get_privacy, pattern=r'^privacy:')],
    },
    fallbacks=[cancel_handler],
)

drive_conv = ConversationHandler(
    entry_points=[CommandHandler('upload_drive', cmd_upload_drive)],
    states={
        DRIVE_WAITING_LINK: [MessageHandler(filters.TEXT & ~filters.COMMAND, drive_get_link)],
        DRIVE_WAITING_TITLE: [MessageHandler(filters.TEXT & ~filters.COMMAND, drive_get_title)],
        DRIVE_WAITING_DESC: [MessageHandler(filters.TEXT & ~filters.COMMAND, drive_get_desc)],
        DRIVE_WAITING_PRIVACY: [CallbackQueryHandler(drive_get_privacy, pattern=r'^privacy:')],
    },
    fallbacks=[cancel_handler],
)

batch_conv = ConversationHandler(
    entry_points=[CommandHandler('batch_drive', cmd_batch_drive)],
    states={
        BATCH_WAITING_LINK: [MessageHandler(filters.TEXT & ~filters.COMMAND, batch_get_link)],
        BATCH_WAITING_PRIVACY: [CallbackQueryHandler(batch_get_privacy, pattern=r'^privacy:')],
    },
    fallbacks=[cancel_handler],
)

app = Application.builder().token(TELEGRAM_BOT_TOKEN).build()
app.add_handler(CommandHandler('start', cmd_start))
app.add_handler(CommandHandler('quota', cmd_quota))
app.add_handler(url_conv)
app.add_handler(drive_conv)
app.add_handler(batch_conv)

print('🤖 Bot is starting! Message it on Telegram.')
print('   Press the ⏹️ stop button to shut down.')

# Colab runs its own event loop, so we use nest_asyncio to allow nested loops
import nest_asyncio
nest_asyncio.apply()
app.run_polling(drop_pending_updates=True)